# Capítulo 10 — Crear nuevas columnas en un DataFrame

## Breve repaso

En el capítulo anterior sobre valores extremos y rankings aprendimos a identificar registros destacados dentro de un `DataFrame`.

Usamos `head()` y `tail()` para observar los primeros y últimos registros, combinamos `sort_values()` con esas herramientas y luego incorporamos métodos más directos como `nlargest()` y `nsmallest()`.

Ese recorrido nos permitió responder preguntas como cuáles son los productos más caros, cuáles son los más baratos, qué ventas tuvieron mayor cantidad vendida y cuáles tuvieron menor cantidad vendida.

Sin embargo, también apareció una idea importante: no siempre las columnas originales del dataset contienen directamente toda la información que necesitamos analizar.

En nuestro dataset tenemos una columna `precio`, que indica el precio unitario de cada producto, y una columna `cantidad_vendida`, que indica cuántas unidades se vendieron. Pero si queremos saber cuánto dinero generó cada venta, necesitamos combinar esas dos columnas. Para eso necesitamos crear una nueva columna.

En este capítulo vamos a aprender a generar nuevas variables dentro de un `DataFrame` a partir de columnas existentes. Empezaremos con operaciones matemáticas simples y luego veremos cómo crear columnas usando condiciones.

Al finalizar este capítulo deberías poder:

- Comprender qué significa crear una nueva columna en un `DataFrame`.
- Crear columnas a partir de operaciones entre columnas existentes.
- Crear una columna de importe total usando precio y cantidad vendida.
- Crear columnas con valores constantes.
- Crear columnas a partir de condiciones simples.
- Usar `loc` para asignar valores según una condición.
- Verificar que la nueva columna se haya creado correctamente.
- Reconocer errores frecuentes al crear o modificar columnas.

## Dataset de trabajo

Para aprender a crear nuevas columnas vamos a seguir usando el dataset de ventas de una tienda escolar.

Cada fila representa una venta registrada. Las columnas actuales indican el producto vendido, la categoría, el precio unitario, la cantidad vendida, la sucursal, la forma de pago y la fecha.

Este dataset es especialmente útil para este capítulo porque contiene dos columnas que podemos combinar de manera natural:

```text
precio
cantidad_vendida
```

A partir de esas dos columnas podremos crear una nueva variable: el importe total de cada venta.


In [1]:
import pandas as pd

datos = {
    "producto": [
        "Cuaderno", "Lapicera", "Mochila", "Regla", "Cartuchera",
        "Calculadora", "Auriculares", "Resaltadores", "Guardapolvo", "Compas",
        "Pendrive", "Carpeta", "Goma", "Lapiz", "Agenda"
    ],
    "categoria": [
        "Libreria", "Libreria", "Accesorios", "Libreria", "Accesorios",
        "Tecnologia", "Tecnologia", "Libreria", "Indumentaria", "Libreria",
        "Tecnologia", "Libreria", "Libreria", "Libreria", "Libreria"
    ],
    "precio": [
        1200, 350, 8500, 500, 3200,
        18500, 7600, 2100, 14500, 2600,
        9800, 950, 300, 250, 4300
    ],
    "cantidad_vendida": [
        15, 40, 5, 25, 8,
        3, 6, 12, 4, 10,
        7, 18, 30, 50, 9
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Norte",
        "Centro", "Sur", "Centro", "Norte", "Sur",
        "Centro", "Norte", "Sur", "Centro", "Sur"
    ],
    "forma_pago": [
        "Efectivo", "Debito", "Credito", "Efectivo", "Debito",
        "Credito", "Credito", "Debito", "Credito", "Efectivo",
        "Debito", "Efectivo", "Efectivo", "Debito", "Credito"
    ],
    "fecha": [
        "2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03",
        "2024-03-03", "2024-03-04", "2024-03-04", "2024-03-05", "2024-03-05",
        "2024-03-06", "2024-03-06", "2024-03-07", "2024-03-07", "2024-03-08"
    ]
}

df = pd.DataFrame(datos)

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05


Al observar el `DataFrame`, vemos que todavía no existe una columna que indique el importe total de cada venta.

Sabemos cuánto cuesta cada unidad y cuántas unidades se vendieron, pero el dataset no trae directamente calculado el resultado de multiplicar esos dos valores.

Ese es un caso muy común en el trabajo con datos. Muchas veces necesitamos crear nuevas columnas para representar información que no aparece explícitamente en el dataset original, pero que puede obtenerse a partir de las columnas disponibles.

## Qué significa crear una nueva columna

Crear una nueva columna significa agregar una variable al `DataFrame`.

Esa nueva variable puede construirse de distintas maneras. A veces surge de una operación matemática entre columnas existentes. Otras veces se crea a partir de una condición, una clasificación, una transformación de texto, una fecha o una regla definida por quien analiza los datos.

En nuestro caso, tenemos dos columnas numéricas:

```text
precio
cantidad_vendida
```

La columna `precio` indica el precio unitario del producto. La columna `cantidad_vendida` indica cuántas unidades se vendieron. Si multiplicamos ambos valores, obtenemos el importe total de esa venta.

Podemos pensarlo así:

```text
importe_total = precio × cantidad_vendida
```

Esa nueva columna no estaba en el dataset original, pero podemos construirla a partir de información que ya tenemos.

Crear columnas nuevas es una tarea muy importante en análisis de datos, porque muchas preguntas no se responden directamente con las variables originales. A veces necesitamos construir una variable más útil para el análisis.

Por ejemplo, con el dataset actual podríamos preguntarnos:

```text
¿Qué venta generó más dinero?
¿Qué categoría produjo mayor importe total?
¿Qué sucursal registró ventas de mayor valor?
```

Para responder esas preguntas, no alcanza con mirar solo el precio o solo la cantidad vendida. Necesitamos combinar ambas columnas en una nueva variable.

## Crear una columna a partir de otras columnas

Para crear una nueva columna en Pandas, escribimos el nombre del `DataFrame`, seguido del nombre de la nueva columna entre corchetes. Por ejemplo, si queremos crear una columna llamada `importe_total`, podemos escribir:

```python
df["importe_total"]
```

Pero además necesitamos indicar qué valores tendrá esa columna. En este caso, el importe total de cada venta se obtiene multiplicando el precio unitario por la cantidad vendida.

In [2]:
df["importe_total"] = df["precio"] * df["cantidad_vendida"]

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha,importe_total
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01,18000
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01,14000
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02,42500
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02,12500
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03,25600
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03,55500
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04,45600
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04,25200
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05,58000
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05,26000


Ahora el `DataFrame` tiene una nueva columna llamada `importe_total`.

Pandas realizó la multiplicación fila por fila. Es decir, tomó el precio de cada producto y lo multiplicó por la cantidad vendida en esa misma fila. Por ejemplo, si una fila tiene:

```text
precio = 1200
cantidad_vendida = 15
```

entonces el importe total será:

```text
1200 × 15 = 18000
```

Esta forma de trabajo es muy común en Pandas. No necesitamos recorrer las filas una por una con un bucle. Cuando escribimos una operación entre columnas, Pandas aplica esa operación de manera vectorizada, es decir, sobre todos los valores de la columna al mismo tiempo.

La nueva columna queda incorporada al `DataFrame`. A partir de este momento podemos seleccionarla, ordenarla, filtrarla o usarla en nuevos cálculos.

## Verificar la nueva columna

Después de crear una columna nueva, conviene verificar que se haya creado correctamente.

Una forma sencilla de hacerlo es observar solamente las columnas involucradas en el cálculo. En este caso, nos interesa revisar `producto`, `precio`, `cantidad_vendida` e `importe_total`.

In [3]:
df[["producto", "precio", "cantidad_vendida", "importe_total"]]

,producto,precio,cantidad_vendida,importe_total
0,Cuaderno,1200,15,18000
1,Lapicera,350,40,14000
2,Mochila,8500,5,42500
3,Regla,500,25,12500
4,Cartuchera,3200,8,25600
5,Calculadora,18500,3,55500
6,Auriculares,7600,6,45600
7,Resaltadores,2100,12,25200
8,Guardapolvo,14500,4,58000
9,Compas,2600,10,26000


Esta vista nos permite comprobar si el cálculo tiene sentido.

Por ejemplo, para cada fila podemos revisar mentalmente que el importe total corresponda a la multiplicación entre el precio unitario y la cantidad vendida.

También podemos usar algunas herramientas que ya conocemos. Por ejemplo, podemos ordenar el `DataFrame` según la nueva columna `importe_total` para ver cuáles fueron las ventas de mayor valor.

In [4]:
df.sort_values("importe_total", ascending=False)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha,importe_total
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06,68600
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05,58000
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03,55500
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04,45600
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02,42500
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08,38700
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05,26000
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03,25600
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04,25200
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01,18000


Ahora el ordenamiento se realiza usando una columna que no estaba en el dataset original, sino que fue creada por nosotros.

Esto muestra una idea importante: una vez que agregamos una columna al `DataFrame`, Pandas puede trabajar con ella igual que con cualquier otra columna. Podemos seleccionarla, ordenarla, filtrarla, resumirla o combinarla con otras variables.

Por ejemplo, podemos mostrar solamente algunas columnas para leer mejor el resultado:

In [5]:
df.sort_values("importe_total", ascending=False)[
    ["producto", "categoria", "precio", "cantidad_vendida", "importe_total"]
]

,producto,categoria,precio,cantidad_vendida,importe_total
10,Pendrive,Tecnologia,9800,7,68600
8,Guardapolvo,Indumentaria,14500,4,58000
5,Calculadora,Tecnologia,18500,3,55500
6,Auriculares,Tecnologia,7600,6,45600
2,Mochila,Accesorios,8500,5,42500
14,Agenda,Libreria,4300,9,38700
9,Compas,Libreria,2600,10,26000
4,Cartuchera,Accesorios,3200,8,25600
7,Resaltadores,Libreria,2100,12,25200
0,Cuaderno,Libreria,1200,15,18000


Esta tabla permite comparar el precio unitario, la cantidad vendida y el importe total de cada venta.

Al observarla, podemos notar que la venta de mayor importe total no necesariamente corresponde al producto con mayor precio unitario ni al producto con mayor cantidad vendida por separado. El importe total depende de la combinación de ambas variables.

## Comparar precio, cantidad e importe total

La nueva columna `importe_total` nos permite analizar las ventas desde una perspectiva más completa.

Antes podíamos construir rankings usando `precio` o `cantidad_vendida`. Pero cada una de esas columnas muestra solo una parte del problema.

El precio indica cuánto cuesta una unidad del producto. La cantidad vendida indica cuántas unidades se vendieron. El importe total combina ambas cosas. Comparemos algunos rankings.

In [6]:
# Ventas con mayor precio unitario

df.nlargest(5, "precio")[
    ["producto", "categoria", "precio", "cantidad_vendida", "importe_total"]
]

,producto,categoria,precio,cantidad_vendida,importe_total
5,Calculadora,Tecnologia,18500,3,55500
8,Guardapolvo,Indumentaria,14500,4,58000
10,Pendrive,Tecnologia,9800,7,68600
2,Mochila,Accesorios,8500,5,42500
6,Auriculares,Tecnologia,7600,6,45600


In [7]:
# Ventas con mayor cantidad vendida

df.nlargest(5, "cantidad_vendida")[
    ["producto", "categoria", "precio", "cantidad_vendida", "importe_total"]
]

,producto,categoria,precio,cantidad_vendida,importe_total
13,Lapiz,Libreria,250,50,12500
1,Lapicera,Libreria,350,40,14000
12,Goma,Libreria,300,30,9000
3,Regla,Libreria,500,25,12500
11,Carpeta,Libreria,950,18,17100


In [8]:
# Ventas con mayor importe total

df.nlargest(5, "importe_total")[
    ["producto", "categoria", "precio", "cantidad_vendida", "importe_total"]
]

,producto,categoria,precio,cantidad_vendida,importe_total
10,Pendrive,Tecnologia,9800,7,68600
8,Guardapolvo,Indumentaria,14500,4,58000
5,Calculadora,Tecnologia,18500,3,55500
6,Auriculares,Tecnologia,7600,6,45600
2,Mochila,Accesorios,8500,5,42500


Al comparar estos tres rankings, podemos observar que no responden la misma pregunta.

El primer ranking muestra los productos con mayor precio unitario. El segundo muestra las ventas con mayor cantidad de unidades. El tercero muestra las ventas que generaron mayor importe total.

Esta comparación es muy importante porque nos recuerda que la variable elegida cambia la interpretación del análisis. Un producto caro puede vender pocas unidades. Un producto barato puede vender muchas unidades. El importe total permite combinar esas dos dimensiones y observar el valor económico de cada venta.

Crear nuevas columnas nos permite construir variables más cercanas a la pregunta que realmente queremos responder.

## Crear una columna con un valor constante

No siempre creamos columnas nuevas a partir de cálculos. A veces necesitamos agregar una columna que tenga el mismo valor para todas las filas. Por ejemplo, podríamos querer registrar que todos los datos de este `DataFrame` pertenecen a una misma tienda o a una misma fuente de información.

Supongamos que queremos agregar una columna llamada `origen` con el valor `"Tienda escolar"`.

In [9]:
df["origen"] = "Tienda escolar"

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha,importe_total,origen
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01,18000,Tienda escolar
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01,14000,Tienda escolar
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02,42500,Tienda escolar
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02,12500,Tienda escolar
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03,25600,Tienda escolar
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03,55500,Tienda escolar
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04,45600,Tienda escolar
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04,25200,Tienda escolar
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05,58000,Tienda escolar
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05,26000,Tienda escolar


Pandas agrega la columna `origen` y coloca el mismo valor en todas las filas.

Este tipo de columna puede ser útil cuando combinamos datos de distintas fuentes. Por ejemplo, si más adelante tuviéramos ventas de varias tiendas, podríamos agregar una columna que indique de dónde proviene cada registro.

Aunque este ejemplo es simple, muestra una idea importante: para crear una columna nueva, solo necesitamos asignar valores a un nombre de columna que todavía no existe.

La estructura general es:

```python
df["nueva_columna"] = valores
```

Esos valores pueden ser un único valor constante, una operación entre columnas, una condición o una transformación más elaborada.

## Crear una columna a partir de una condición

También podemos crear columnas nuevas usando condiciones.

Por ejemplo, ahora que tenemos la columna `importe_total`, podríamos clasificar cada venta según su valor. Una forma simple sería marcar si una venta tuvo un importe alto o no.

Supongamos que queremos considerar como venta de importe alto a toda venta cuyo `importe_total` sea mayor o igual que `50000`.

Primero podemos construir la condición:

In [10]:
df["importe_total"] >= 50000

,importe_total
0,False
1,False
2,False
3,False
4,False
5,True
6,False
7,False
8,True
9,False


La salida es una máscara booleana. Cada fila recibe `True` si su importe total es mayor o igual que `50000`, y `False` si no lo es.

Ahora podemos usar esa condición para crear una nueva columna llamada `venta_alta`.

In [11]:
df["venta_alta"] = df["importe_total"] >= 50000

df[["producto", "precio", "cantidad_vendida", "importe_total", "venta_alta"]]

,producto,precio,cantidad_vendida,importe_total,venta_alta
0,Cuaderno,1200,15,18000,False
1,Lapicera,350,40,14000,False
2,Mochila,8500,5,42500,False
3,Regla,500,25,12500,False
4,Cartuchera,3200,8,25600,False
5,Calculadora,18500,3,55500,True
6,Auriculares,7600,6,45600,False
7,Resaltadores,2100,12,25200,False
8,Guardapolvo,14500,4,58000,True
9,Compas,2600,10,26000,False


La nueva columna `venta_alta` contiene valores booleanos: `True` o `False`.

Esto significa que cada fila quedó clasificada según una condición. Las ventas con importe total mayor o igual que `50000` aparecen como `True`. Las demás aparecen como `False`.

Este tipo de columna puede ser útil cuando queremos marcar registros que cumplen cierto criterio. Más adelante podremos usar columnas como esta para filtrar, contar o comparar grupos.

## Crear una columna con etiquetas de texto

La columna `venta_alta` que creamos recién contiene valores booleanos: `True` o `False`.

Eso puede ser útil, pero a veces queremos que la nueva columna tenga etiquetas más descriptivas. Por ejemplo, en lugar de mostrar `True` o `False`, podríamos crear una columna que diga `"Alta"` o `"Normal"`.

Para resolver este tipo de situación podemos usar `np.where()`.

`np.where()` es una función de NumPy que permite elegir entre dos valores según se cumpla o no una condición. En este caso, la vamos a usar para decir:

```text
si la venta cumple la condición → escribir "Alta"
si no cumple la condición       → escribir "Normal"
```

NumPy es una biblioteca muy usada en Python para trabajar con datos numéricos y operaciones sobre conjuntos de valores. Pandas se apoya muchas veces en NumPy, y por eso es habitual usarlas juntas.

Para poder usar `np.where()`, primero importamos NumPy con el alias `np`.


In [12]:
import numpy as np

Ahora podemos crear una columna llamada `tipo_venta`.

La regla será:

```text
si importe_total es mayor o igual que 50000 → "Alta"
si no                                       → "Normal"
```

Esa regla tiene tres partes: una condición, un valor para los casos donde la condición se cumple y otro valor para los casos donde la condición no se cumple.

In [13]:
df["tipo_venta"] = np.where(
    df["importe_total"] >= 50000,
    "Alta",
    "Normal"
)

df[["producto", "importe_total", "venta_alta", "tipo_venta"]]

,producto,importe_total,venta_alta,tipo_venta
0,Cuaderno,18000,False,Normal
1,Lapicera,14000,False,Normal
2,Mochila,42500,False,Normal
3,Regla,12500,False,Normal
4,Cartuchera,25600,False,Normal
5,Calculadora,55500,True,Alta
6,Auriculares,45600,False,Normal
7,Resaltadores,25200,False,Normal
8,Guardapolvo,58000,True,Alta
9,Compas,26000,False,Normal


La nueva columna `tipo_venta` clasifica cada registro usando una etiqueta de texto.

Cuando la condición `df["importe_total"] >= 50000` se cumple, `np.where()` devuelve el valor `"Alta"`. Cuando la condición no se cumple, devuelve el valor `"Normal"`. Ese resultado se guarda en la nueva columna `tipo_venta`.

Esta forma de trabajo es útil cuando queremos transformar una condición en una categoría más fácil de leer.

Podemos pensar la estructura de `np.where()` de esta manera:

```python
np.where(condición, valor_si_verdadero, valor_si_falso)
```

En nuestro ejemplo:

```python
np.where(
    df["importe_total"] >= 50000,
    "Alta",
    "Normal"
)
```

significa: si el importe total es mayor o igual que `50000`, escribí `"Alta"`; en caso contrario, escribí `"Normal"`.

Este tipo de clasificación aparece con frecuencia en el trabajo con datos. Nos permite crear categorías nuevas a partir de reglas definidas por el análisis.

## Asignar valores usando loc

Otra forma de crear o modificar columnas según una condición es usar `loc`.

Esta opción resulta muy útil cuando queremos asignar valores solo a las filas que cumplen cierto criterio.

Por ejemplo, podríamos crear una columna llamada `observacion` y colocar inicialmente el texto `"Sin observaciones"` para todas las filas.

In [14]:
df["observacion"] = "Sin observaciones"

df[["producto", "importe_total", "observacion"]]

,producto,importe_total,observacion
0,Cuaderno,18000,Sin observaciones
1,Lapicera,14000,Sin observaciones
2,Mochila,42500,Sin observaciones
3,Regla,12500,Sin observaciones
4,Cartuchera,25600,Sin observaciones
5,Calculadora,55500,Sin observaciones
6,Auriculares,45600,Sin observaciones
7,Resaltadores,25200,Sin observaciones
8,Guardapolvo,58000,Sin observaciones
9,Compas,26000,Sin observaciones


Ahora todas las filas tienen el mismo valor en la columna `observacion`.

Supongamos que queremos cambiar ese texto solamente para las ventas cuyo importe total sea mayor o igual que `50000`. En esas filas vamos a escribir `"Revisar venta alta"`.

In [15]:
df.loc[
    df["importe_total"] >= 50000,
    "observacion"
] = "Revisar venta alta"

df[["producto", "importe_total", "tipo_venta", "observacion"]]

,producto,importe_total,tipo_venta,observacion
0,Cuaderno,18000,Normal,Sin observaciones
1,Lapicera,14000,Normal,Sin observaciones
2,Mochila,42500,Normal,Sin observaciones
3,Regla,12500,Normal,Sin observaciones
4,Cartuchera,25600,Normal,Sin observaciones
5,Calculadora,55500,Alta,Revisar venta alta
6,Auriculares,45600,Normal,Sin observaciones
7,Resaltadores,25200,Normal,Sin observaciones
8,Guardapolvo,58000,Alta,Revisar venta alta
9,Compas,26000,Normal,Sin observaciones


En este caso usamos `loc` para indicar dos cosas al mismo tiempo.

Primero indicamos qué filas queremos modificar:

```python
df["importe_total"] >= 50000
```

Después indicamos qué columna queremos modificar:

```python
"observacion"
```

La estructura general es:

```python
df.loc[condición, "columna"] = nuevo_valor
```

Esta forma de trabajo permite modificar solo una parte del `DataFrame`. Las filas que cumplen la condición reciben el nuevo valor. Las filas que no cumplen la condición conservan el valor anterior.

Usar `loc` para asignar valores es una práctica muy frecuente en Pandas, especialmente cuando queremos crear categorías, marcar casos especiales o corregir valores en una columna.

## Revisar las columnas creadas

Después de crear varias columnas nuevas, conviene revisar cómo quedó el `DataFrame`.

Hasta ahora agregamos estas columnas:

```text
importe_total
origen
venta_alta
tipo_venta
observacion
```

Podemos consultar los nombres de todas las columnas usando `columns`.

In [16]:
df.columns

Index(['producto', 'categoria', 'precio', 'cantidad_vendida', 'sucursal',
       'forma_pago', 'fecha', 'importe_total', 'origen', 'venta_alta',
       'tipo_venta', 'observacion'],
      dtype='object')

También podemos observar una vista reducida del `DataFrame` para concentrarnos solamente en las columnas más importantes para este capítulo.

In [17]:
df[
    [
        "producto",
        "precio",
        "cantidad_vendida",
        "importe_total",
        "tipo_venta",
        "observacion"
    ]
]

,producto,precio,cantidad_vendida,importe_total,tipo_venta,observacion
0,Cuaderno,1200,15,18000,Normal,Sin observaciones
1,Lapicera,350,40,14000,Normal,Sin observaciones
2,Mochila,8500,5,42500,Normal,Sin observaciones
3,Regla,500,25,12500,Normal,Sin observaciones
4,Cartuchera,3200,8,25600,Normal,Sin observaciones
5,Calculadora,18500,3,55500,Alta,Revisar venta alta
6,Auriculares,7600,6,45600,Normal,Sin observaciones
7,Resaltadores,2100,12,25200,Normal,Sin observaciones
8,Guardapolvo,14500,4,58000,Alta,Revisar venta alta
9,Compas,2600,10,26000,Normal,Sin observaciones


Esta vista nos permite revisar si las nuevas columnas tienen sentido.

La columna `importe_total` surge de multiplicar `precio` por `cantidad_vendida`. La columna `venta_alta` indica con `True` o `False` si una venta supera cierto umbral. La columna `tipo_venta` expresa una clasificación similar, pero con etiquetas de texto. La columna `observacion` agrega una descripción para las ventas que queremos destacar.

También podemos usar `info()` para revisar la estructura general del `DataFrame` después de agregar columnas.

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   producto          15 non-null     object
 1   categoria         15 non-null     object
 2   precio            15 non-null     int64 
 3   cantidad_vendida  15 non-null     int64 
 4   sucursal          15 non-null     object
 5   forma_pago        15 non-null     object
 6   fecha             15 non-null     object
 7   importe_total     15 non-null     int64 
 8   origen            15 non-null     object
 9   venta_alta        15 non-null     bool  
 10  tipo_venta        15 non-null     object
 11  observacion       15 non-null     object
dtypes: bool(1), int64(3), object(8)
memory usage: 1.4+ KB


Al mirar la salida de `info()`, veremos que el `DataFrame` ahora tiene más columnas que al comienzo.

Esto muestra que crear columnas modifica la estructura del `DataFrame`. Por eso es importante revisar el resultado después de cada transformación relevante.

En análisis de datos, no alcanza con escribir una instrucción y seguir adelante. Cada vez que modificamos el dataset, conviene verificar que el cambio haya producido el resultado esperado.

## Errores frecuentes al crear columnas

Al crear nuevas columnas en Pandas, pueden aparecer algunos errores o confusiones frecuentes.

Una primera confusión es escribir mal el nombre de una columna existente. Por ejemplo, para crear `importe_total` usamos las columnas `precio` y `cantidad_vendida`. Si escribimos mal uno de esos nombres, Pandas no podrá encontrar la columna.

In [19]:
# Esta instrucción produciría un error porque la columna se llama "precio", no "Precio".
# df["importe_total_error"] = df["Precio"] * df["cantidad_vendida"]

Para evitar este problema, conviene revisar los nombres disponibles con:

In [20]:
df.columns

Index(['producto', 'categoria', 'precio', 'cantidad_vendida', 'sucursal',
       'forma_pago', 'fecha', 'importe_total', 'origen', 'venta_alta',
       'tipo_venta', 'observacion'],
      dtype='object')

Otra confusión frecuente es creer que una nueva columna se crea de manera temporal. En realidad, cuando escribimos una asignación como:

In [21]:
df["importe_total"] = df["precio"] * df["cantidad_vendida"]

la columna queda agregada al `DataFrame`. A partir de ese momento, forma parte de `df` y puede usarse en selecciones, filtros, ordenamientos o nuevos cálculos.

También debemos tener cuidado al crear columnas usando condiciones. Por ejemplo, si definimos una venta alta como una venta con importe total mayor o igual que `50000`, esa regla depende de una decisión nuestra. Otro análisis podría usar un umbral diferente, como `30000`, `75000` o el promedio de los importes.

Por eso, cuando creamos columnas a partir de reglas, conviene que el nombre de la columna sea claro y que el criterio usado esté explicado.

Otro punto importante es no crear demasiadas columnas sin necesidad. Agregar columnas puede enriquecer el análisis, pero también puede volver el `DataFrame` más difícil de leer si no tenemos claro para qué sirve cada nueva variable.

Una buena práctica es revisar el `DataFrame` después de cada transformación importante y conservar nombres de columnas descriptivos.

Por ejemplo, un nombre como `importe_total` es más claro que un nombre como `total`, `valor`, `dato_nuevo` o `columna_1`.

Al crear columnas conviene verificar tres cosas:

```text
que los nombres de columnas usados existan
que la operación o condición represente lo que queremos calcular
que la nueva columna tenga un nombre claro
```

Crear columnas es una herramienta muy poderosa, pero también implica tomar decisiones sobre cómo queremos representar la información.


## Resumen del capítulo

En este capítulo aprendimos a crear nuevas columnas dentro de un `DataFrame`.

La idea central fue que muchas veces los datos originales no contienen directamente toda la información que necesitamos analizar. En esos casos, podemos construir nuevas variables a partir de columnas existentes, operaciones matemáticas, condiciones o reglas definidas durante el análisis.

Primero creamos la columna `importe_total`, multiplicando el precio unitario por la cantidad vendida:

```python
df["importe_total"] = df["precio"] * df["cantidad_vendida"]
```

Esta nueva columna nos permitió responder preguntas que no podían resolverse mirando solamente `precio` o `cantidad_vendida` por separado. A partir de `importe_total`, pudimos ordenar ventas, construir rankings y observar cuáles generaron mayor valor económico.

También vimos que, una vez creada, una columna nueva puede usarse igual que cualquier otra columna del `DataFrame`. Podemos seleccionarla, ordenarla, filtrarla, mostrarla junto con otras variables o usarla para nuevos cálculos.

Después creamos una columna con un valor constante:

```python
df["origen"] = "Tienda escolar"
```

Este tipo de columna puede ser útil cuando queremos registrar una fuente, una etiqueta general o un dato común a todas las filas.

Más adelante creamos columnas a partir de condiciones. Primero generamos una columna booleana:

```python
df["venta_alta"] = df["importe_total"] >= 50000
```

Luego usamos `np.where()` para crear una columna con etiquetas de texto:

```python
df["tipo_venta"] = np.where(
    df["importe_total"] >= 50000,
    "Alta",
    "Normal"
)
```

También usamos `loc` para asignar valores solamente a las filas que cumplían una condición:

```python
df.loc[
    df["importe_total"] >= 50000,
    "observacion"
] = "Revisar venta alta"
```

Esta forma de trabajo permite modificar una parte específica del `DataFrame` sin afectar todas las filas.

Finalmente revisamos las columnas creadas usando herramientas como `columns`, selección de columnas e `info()`. Esa verificación es importante porque cada nueva columna modifica la estructura del dataset.

La idea principal de este capítulo fue:

```text
Crear columnas nuevas permite construir información que no estaba explícita en el dataset original.
```

Crear columnas es una de las herramientas más importantes del trabajo con datos. Nos permite transformar un dataset inicial en una versión más útil para responder preguntas de análisis.

## Próximo paso

Ya sabemos seleccionar, filtrar, ordenar, construir rankings y crear nuevas columnas.

El siguiente paso será aprender a modificar columnas existentes. Muchas veces no necesitamos agregar una variable nueva, sino corregir, transformar o ajustar una columna que ya existe.

Por ejemplo, podemos querer cambiar nombres de categorías, normalizar textos, reemplazar valores, modificar formatos o preparar una columna para que sea más fácil de analizar.

Ese trabajo nos acercará cada vez más a una etapa fundamental del análisis de datos: la limpieza y preparación del dataset.